# Steering on the dataset's own topics, and which layer to steer (Qwen3-8B, GPU)

Two experiments that use Praveen's dataset instead of made-up prompts:

1. **Topic sweep:** each of the dataset's 20 topics becomes a neutral prompt ("Write a short paragraph about this situation: A character waits for an important phone call."). Strengths 0, 0.25, 0.5, 1.0 and 1.5 with the desperate vector at block 23, plus a second unsteered reply as a control. Topics 0–13 are the ones whose stories built the vector; topics 14–19 never touched it.
2. **Layer sweep:** the same 20 prompts at strength 0.5, steering at blocks 5, 11, 17, 23, 29 or 35, each with its own block's desperate vector and residual size. This tests the "two-thirds of the way in" choice with data.

Outputs: `results/steering/sweep_topics_desperate.json` and `results/steering/layer_sweep_desperate.json`, scored afterwards by `judge_more.ipynb`.

**Run on:** an NVIDIA GPU, from inside `notebooks/`, with the same files as `steering.ipynb`.

In [1]:
import json, sys, time
from pathlib import Path
import numpy as np
import torch

sys.path.insert(0, str(Path.cwd().parent))
from src.steering import LAYER, MODEL_REVISION, load_model_8b, residual_norms, steer, generate

RUN_DIR = Path("../emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c")
STORIES = Path("../emotion_vector_pipeline/data/emotion_stories_qwen_full1120.jsonl")
OUT = Path("../results/steering")
EMOTION = "desperate"
STRENGTHS = [0, 0.25, 0.5, 1.0, 1.5]
LAYERS = [5, 11, 17, 23, 29, 35]

rows = [json.loads(line) for line in STORIES.open()]
topics = {r["topic_id"]: (r["topic"], r["split"]) for r in rows}
PROMPTS = [f"Write a short paragraph about this situation: {topics[t][0]}." for t in range(20)]
print(PROMPTS[8])

with np.load(RUN_DIR / "emotion_vectors.npz", allow_pickle=False) as saved:
    names = saved["emotion_names"].tolist()
    assert saved["layer_indices"].tolist() == list(range(36))
    UNIT = torch.tensor(saved["unit_vectors"][names.index(EMOTION)])  #one desperate vector per block: [36, 4096]
print(UNIT.shape)  #expect torch.Size([36, 4096])

Write a short paragraph about this situation: A character waits for an important phone call.
torch.Size([36, 4096])


In [2]:
tok, model = load_model_8b()

#residual size at every block we steer, measured on the same 28 validation stories as steering.ipynb
texts = []
for e in sorted({r["emotion"] for r in rows}):
    texts += [r["text"] for r in rows if r["emotion"] == e and r["split"] == "validation"][:2]
NORM = {}
for L in LAYERS:
    _, rest = residual_norms(model, tok, texts, layer=L)
    NORM[L] = rest.mean().item()
print({L: round(n, 1) for L, n in NORM.items()})  #block 23 should match steering.ipynb (198.0)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

{5: 38.0, 11: 63.1, 17: 88.6, 23: 198.0, 29: 543.8, 35: 1297.6}


## 1. Topic sweep at block 23

In [3]:
records, start = [], time.time()
for t, prompt in enumerate(PROMPTS):
    seed = 2000 + t
    rec = dict(prompt=prompt, topic_id=t, topic_split=topics[t][1], seed=seed, outputs={})
    for c in STRENGTHS:
        with steer(model, UNIT[LAYER], c, NORM[LAYER]):
            rec["outputs"][str(c)] = generate(model, tok, prompt, seed=seed, max_new_tokens=100)
    rec["baseline_other_seed"] = generate(model, tok, prompt, seed=seed + 500, max_new_tokens=100)
    records.append(rec)
    print(f"{t + 1}/20 done, {time.time() - start:.0f}s")

#strength 0 adds a zero vector, so it must match a run with no hook
assert records[0]["outputs"]["0"] == generate(model, tok, PROMPTS[0], seed=2000, max_new_tokens=100)
sweep = dict(emotion=EMOTION, layer=LAYER, residual_norm=NORM[LAYER], model_revision=MODEL_REVISION,
             strengths=STRENGTHS, max_new_tokens=100, temperature=0.7, top_p=0.9, prompt_set="dataset topics",
             records=records)
(OUT / "sweep_topics_desperate.json").write_text(json.dumps(sweep, indent=2))
for c in STRENGTHS:
    print(f"--- {c} ---\n{records[8]['outputs'][str(c)]}\n")

1/20 done, 33s


2/20 done, 64s


3/20 done, 96s


4/20 done, 129s


5/20 done, 162s


6/20 done, 194s


... (11 outputs left out)

18/20 done, 575s


19/20 done, 607s


20/20 done, 637s


--- 0 ---
She sat on the edge of her bed, fingers curled around her phone, heart pounding in her chest. The minutes dragged on, each second stretching like an eternity. She had waited for this call all morning, and now, with the clock ticking toward the appointed time, her nerves were on edge. The silence in the room felt heavy, filled with the weight of what could happen next. She kept checking the time, hoping against hope that the call would come soon—because this was more than just

--- 0.25 ---
She sat on the edge of her bed, staring at her phone, her hands trembling. The call was supposed to change everything—her future, her life. Every ring echoed in her ears, but it wasn't coming. She kept checking the time, hoping she wasn't too late. Her heart pounded, and she forced herself to breathe. This was it. The call that could save her or destroy her. She couldn't let it go to voicemail.

--- 0.5 ---
She's been waiting for hours, sitting on the floor with her phone, shaking. The call

## 2. Layer sweep at strength 0.5

In [4]:
layer_records, start = [], time.time()
for t, prompt in enumerate(PROMPTS):
    rec = dict(prompt=prompt, topic_id=t, seed=2000 + t, outputs={"none": records[t]["outputs"]["0"]})
    for L in LAYERS:
        with steer(model, UNIT[L], 0.5, NORM[L], layer=L):
            rec["outputs"][str(L)] = generate(model, tok, prompt, seed=2000 + t, max_new_tokens=100)
    layer_records.append(rec)
    print(f"{t + 1}/20 done, {time.time() - start:.0f}s")

layer_sweep = dict(emotion=EMOTION, strength=0.5, layers=LAYERS, residual_norms=NORM,
                   model_revision=MODEL_REVISION, records=layer_records)
(OUT / "layer_sweep_desperate.json").write_text(json.dumps(layer_sweep, indent=2))
for L in LAYERS:
    print(f"--- block {L} ---\n{layer_records[8]['outputs'][str(L)]}\n")

1/20 done, 29s


2/20 done, 58s


3/20 done, 88s


4/20 done, 119s


5/20 done, 151s


6/20 done, 183s


... (10 outputs left out)

17/20 done, 525s


18/20 done, 557s


19/20 done, 587s


20/20 done, 620s
--- block 5 ---
She sits by the window, fingers curled around the phone, heart pounding in her chest, waiting for the call that could change everything. Every ring is a reminder of how much is on the line—news, a decision, a future. Time ticks by, but she can't move. The silence is louder than the ringing. She checks the time again. She checks the time again. She's ready. She's waiting.

--- block 11 ---
The character sits on the edge of their bed, heart pounding, fingers trembling as they stare at the phone. They've been waiting for this call for hours. It's the one that could change everything—news about their promotion, their future, maybe even their life. The clock ticks louder than the silence in the room. They can't afford to miss it. The phone buzzes, but it's not the call yet. They take a deep breath, trying to stay calm. What if it

--- block 17 ---
She sat on the edge of her bed, her hands trembling as she stared at her phone. The minutes felt like hours. Eve